# Dataset for Modeling

## Libraries

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent ))

In [2]:
from src.dominick import DominickDataLoader, DominickDataSaver
from src.eda import EDA
import numpy as np
import math
import pandas as pd

## Loader

In [3]:
loader = DominickDataLoader()
df = loader.load("dominick_features.csv")

## Filtering
> For our porpuse, we filter those lines that `unit_sold = 0`. Morevoer, for consistency, we filter:
1. total_price > 0.05

In [4]:
df = df[df["units_sold"] != 0]
df = df[df["total_price"] > 0.05]

## EDA

In [5]:
eda = EDA()

#### Uniqueness

In [6]:
# is the grain; store-week-upc unique?
res_grain = eda.analyze_grain_uniqueness(df)
res_grain

,grain_cols,n_rows,n_unique_keys,n_duplicate_keys,pct_duplicate_keys,n_rows_with_missing_in_grain,is_unique_grain
0,"store_code, week_id, upc_code",1966147,1966147,0,0.0,0,True


#### Store-week Coverage

In [7]:
cov_sw = eda.analyze_store_week_coverage(df, min_weeks_for_good=150)
stores_to_drop = cov_sw["low_coverage_stores"]

print("Combinations store-week possible:", cov_sw["n_store_week_possible"])
print("Observed:", cov_sw["n_store_week_observed"])
print("Store-week coverage (%):", cov_sw["store_week_density_pct"])

print(f"Stores to drop (< 150 weeks): {len(stores_to_drop)}")
print(stores_to_drop)

Combinations store-week possible: 26878
Observed: 21004
Store-week coverage (%): 78.1457
Stores to drop (< 150 weeks): 19
[2, 28, 44, 47, 48, 50, 52, 83, 88, 92, 97, 104, 135, 140, 141, 142, 143, 144, 146]


In [8]:
# 2) Quedarte solo con stores que tienen >= 150 weeks
df_filtered = df[~df["store_code"].isin(stores_to_drop)].copy()

print(f"Rows before: {len(df)}")
print(f"Rows after: {len(df_filtered)}")
print(f"Stores before: {df['store_code'].nunique()}")
print(f"Stores after: {df_filtered['store_code'].nunique()}")

Rows before: 1966147
Rows after: 1905978
Stores before: 89
Stores after: 70


#### Store-upc coverage

In [9]:
series_cov = eda.analyze_store_upc_coverage(df_filtered)

summary = series_cov["summary_stats"]
print("n_pairs:", summary["n_pairs"])
print("n_obs quantiles:", summary["n_obs"])
print("coverage_ratio quantiles:", summary["coverage_ratio"])
print("missing_within_span quantiles:", summary["missing_within_span"])

n_pairs: 27390
n_obs quantiles: {'mean': 69.58663745892662, 'p25': 12.0, 'p50': 41.0, 'p75': 109.0, 'p90': 194.0}
coverage_ratio quantiles: {'mean': 0.6235904725026618, 'p25': 0.4, 'p50': 0.6470588235294118, 'p75': 0.8766519823788547, 'p90': 0.9691629955947136}
missing_within_span quantiles: {'mean': 40.527966411098944, 'p25': 8.0, 'p50': 24.0, 'p75': 57.0, 'p90': 107.0}


#### Price Variation

In [10]:
pv = eda.analyze_price_variation(df_filtered, price_col="log_price_per_liter")

summary = pv["summary_stats"]
print("Pairs (store, upc):", summary["n_pairs"])
print("Price levels (n_price_levels):", summary["n_price_levels"])
print("Price changes (n_price_changes):", summary["n_price_changes"])
print("Series with constant price:", summary["n_constant_price_series"], "⇒", summary["pct_constant_price_series"], "%")
print("Price range in log (price_range_log):", summary["price_range_log"])
print("Example p50 ratio max/min aprox.", round(math.exp(summary["price_range_log"]["p50"]), 4))

Pairs (store, upc): 27390
Price levels (n_price_levels): {'mean': 6.531507849580139, 'p25': 2.0, 'p50': 5.0, 'p75': 9.0, 'p90': 15.0}
Price changes (n_price_changes): {'mean': 21.7767433369843, 'p25': 2.0, 'p50': 11.0, 'p75': 32.0, 'p90': 65.0}
Series with constant price: 3859 ⇒ 14.0891 %
Price range in log (price_range_log): {'mean': 0.23329991821108229, 'p25': 0.13393552083779037, 'p50': 0.2519853441061396, 'p75': 0.33705117839279586, 'p90': 0.4062652575120691}
Example p50 ratio max/min aprox. 1.2866


#### Collinearity

In [11]:
pp = eda.analyze_promo_price_collinearity(df_filtered, price_col="log_price_per_liter")

gs = pp["global_stats"]
print("Shares promo:", gs["promo_shares"])
print("on_promo = any(promo_*):", gs["on_promo_matches_any_promo"])
print("Mutually exclusive promos:", gs["promos_mutually_exclusive"])
print("price_promo_gap:", gs["price_promo_gap"])
print("corr(log_price, on_promo):", gs["corr_price_on_promo"])
print("share_changes_with_promo_switch:", gs["share_changes_with_promo_switch"])

Shares promo: {'promo_B': 0.27100732537311556, 'promo_S': 0.005000582378180651, 'promo_C': 0.0001668434787809723}
on_promo = any(promo_*): True
Mutually exclusive promos: True
price_promo_gap: {'mean': -0.15962132304439058, 'median': -0.16794364884684027, 'pct_series_promo_more_expensive': 1.57897341978914}
corr(log_price, on_promo): {'mean': -0.7587414524706965, 'median': -0.8102257102984949}
share_changes_with_promo_switch: {'p50': 0.7142857142857143, 'p75': 0.8350015013512161, 'p90': 1.0}


##### Filtering

In [12]:
min_n_obs = 52
min_coverage = 0.75
min_price_levels = 3
min_price_changes = 5
min_price_range_log = 0.15
max_abs_corr_price_promo = 0.80
max_share_changes_with_promo_switch = 0.80

In [13]:
cov_per_series = series_cov["per_series_coverage"][["store_code", "upc_code", "coverage_ratio"]]
price_per_series = pv["per_series"]
pp_per_series = pp["per_series"][["store_code", "upc_code", "corr_price_on_promo", "share_changes_with_promo_switch"]]

series_stats = (
    price_per_series
    .merge(cov_per_series, on=["store_code", "upc_code"], how="inner")
    .merge(pp_per_series, on=["store_code", "upc_code"], how="left")
)

mask = (
    (series_stats["n_obs"] >= min_n_obs)
    & (series_stats["coverage_ratio"] >= min_coverage)
    & (series_stats["n_price_levels"] >= min_price_levels)
    & (series_stats["n_price_changes"] >= min_price_changes)
    & (series_stats["price_range_log"] >= min_price_range_log)
    & (series_stats["corr_price_on_promo"].abs() <= max_abs_corr_price_promo)
    & (series_stats["share_changes_with_promo_switch"] <= max_share_changes_with_promo_switch)
)

mask = mask & series_stats["corr_price_on_promo"].notna() & series_stats["share_changes_with_promo_switch"].notna()

good_pairs = series_stats.loc[mask, ["store_code", "upc_code"]].drop_duplicates()

print("Nº pairs (store, upc) that pass filters:", len(good_pairs))

# 5) Filtrar df_imputed
df_filtered_prices = df_filtered.merge(
    good_pairs,
    on=["store_code", "upc_code"],
    how="inner",
)

print("Rows df_imputed before:", len(df_filtered))
print("Rows df_imputed after filtering:", len(df_filtered_prices))

Nº pairs (store, upc) that pass filters: 2656
Rows df_imputed before: 1905978
Rows df_imputed after filtering: 484864


#### Outliers

In [14]:
global_out = eda.analyze_outliers(
    df_filtered_prices,
    columns=["log_price_per_liter", "log_liters_sold"],
    method="iqr",
)
print(global_out[["variable", "pct_outliers", "scope"]])

              variable  pct_outliers   scope
0  log_price_per_liter          3.95  global
1      log_liters_sold          0.57  global


In [15]:
by_upc = eda.analyze_outliers(
    df_filtered_prices,
    columns=["log_price_per_liter", "log_liters_sold"],
    method="iqr",
    group_col="upc_code",
)
print(by_upc[["variable", "pct_outliers", "scope"]])

              variable  pct_outliers        scope
0  log_price_per_liter          4.36  by_upc_code
1      log_liters_sold          0.49  by_upc_code


In [16]:
out = eda.analyze_outliers(
    df_filtered_prices,
    columns=["log_price_per_liter", "log_liters_sold"],
    method="iqr",
    group_col="upc_code",
    return_flagged_df=True,
)
summary = out["summary"]
flagged_df = out["flagged_df"]

In [17]:
# Conteos de outliers por columna
print("Total rows:", len(flagged_df))
for c in summary["variable"]:
    flag_col = f"is_outlier_{c}"
    n = flagged_df[flag_col].sum()
    pct = flagged_df[flag_col].mean() * 100
    print(f"  {flag_col}: {int(n)} filas ({pct:.2f}%)")

# Muestra de flagged_df: columnas originales + flags
cols_show = ["store_code", "upc_code", "week_id", "log_price_per_liter", "log_liters_sold"] + [
    c for c in flagged_df.columns if c.startswith("is_outlier_")
]
display(flagged_df[cols_show].head(10))

Total rows: 484864
  is_outlier_log_price_per_liter: 21142 filas (4.36%)
  is_outlier_log_liters_sold: 2396 filas (0.49%)


,store_code,upc_code,week_id,log_price_per_liter,log_liters_sold,is_outlier_log_price_per_liter,is_outlier_log_liters_sold
0,12,1820000051,92,0.305497,0.755790,False,False
1,12,1820000051,93,0.305497,1.854403,False,False
2,12,1820000051,94,0.305497,1.448938,False,False
3,12,1820000051,95,0.305497,0.755790,False,False
4,12,1820000051,96,0.305497,2.142085,False,False
5,12,1820000051,97,0.305497,2.142085,False,False
6,12,1820000051,98,0.305497,1.448938,False,False
7,12,1820000051,99,0.305497,1.448938,False,False
8,12,1820000051,100,0.305497,2.547550,False,False
9,12,1820000051,101,0.305497,1.448938,False,False


In [18]:
df_no_price_out = flagged_df[~flagged_df["is_outlier_log_price_per_liter"]]

print(f"Rows before: {len(df_filtered_prices)}")
print(f"Rows after: {len(df_no_price_out)}")

Rows before: 484864
Rows after: 463722


#### Global Dataset Coverage

In [19]:
cov = eda.analyze_dataset_coverage(df_no_price_out)

print("Row:", cov["n_rows"])
print("Stores:", cov["n_stores"])
print("UPCs:", cov["n_upcs"])
print("Observed weeks:", cov["n_weeks_observed"])
print(f"Week range: {cov['week_min']} … {cov['week_max']}")
print("Global missing weeks (IDs):",cov["weeks_missing"] )

Row: 463722
Stores: 70
UPCs: 203
Observed weeks: 302
Week range: 91 … 399
Global missing weeks (IDs): [219, 262, 263, 264, 265, 284, 285]


#### Span

In [20]:
# Pasar las global_gap_weeks que ya conoces
df_imputed_prices = eda.impute_calendar_gaps(
    df_no_price_out,
    value_cols=['units_sold', 'units_per_deal', 'total_price', 'promo_flag',
       'gross_margin_pct', 'unit_price', 'sales_dolar', 'gross_margin_rate',
       'gross_margin_dolar', 'cost_dolar', 'liters_per_upc', 'price_per_upc',
       'liters_sold', 'price_per_liter', 'on_promo', 'promo_B', 'promo_C',
       'promo_S', 'log_liters_sold', 'log_price_per_liter'],
    global_gap_weeks=cov["weeks_missing"],
)

# Verificar resultado
print("Original rows:", len(df_no_price_out))
print("Rows after imputation:", len(df_imputed_prices))
print("Imputed rows:", df_imputed_prices["is_imputed_calendar_row"].sum())
print("  of which global gap:", df_imputed_prices["is_global_gap_week"].sum())
print("  of which internal gap:", df_imputed_prices["is_internal_gap"].sum())

Original rows: 463722
Rows after imputation: 532965
Imputed rows: 69243
  of which global gap: 15816
  of which internal gap: 53427


##### Columns added by calendar gap imputation

| Column                    | Type      | Description |
|---------------------------|-----------|-------------|
| `is_imputed_calendar_row` | int (0/1) | **1 if the row was created by the calendar imputer**, i.e. it did not exist in the original dataset. 0 for original rows. |
| `is_global_gap_week`      | int (0/1) | **1 if the week was absent across the entire dataset** (no store/UPC has any row for that `week_id`). 0 otherwise. |
| `is_internal_gap`         | int (0/1) | **1 if the week was missing only within this series (store, upc)**, but exists for other series. 0 otherwise. |
| `gap_size_from_prev_obs`  | int       | **Total length of the consecutive missing-week block** this row belongs to. E.g. if weeks 200, 201 and 202 are missing for a (store, upc), all three imputed rows have `gap_size_from_prev_obs = 3`. |
| `weeks_since_last_obs`    | int       | **1-indexed position of this row within its gap block.** Using the example above (200, 201, 202): 200: 1, 201: 2, 202: 3. |

### Quick interpretation

- **Original row**: `is_imputed_calendar_row = 0`, all other flags = 0.
- **Row created by a global gap**: `is_imputed_calendar_row = 1` and `is_global_gap_week = 1`.
- **Row created by a series-internal gap**: `is_imputed_calendar_row = 1` and `is_internal_gap = 1`.

#### Trends

In [21]:
trends = eda.analyze_aggregated_trends(
    df_imputed_prices,
    time_col="week_id",
    price_col="log_price_per_liter",
    demand_raw_col="liters_sold",   
    promo_col="on_promo",
)

print("Correlations week_rank vs aggregated series by week:")
for k, v in trends["correlations"].items():
    print(f"  {k}: {v:.2f}")

Correlations week_rank vs aggregated series by week:
  corr_week_rank_mean_log_price: 0.82
  corr_week_rank_log_total_demand: -0.74
  corr_week_rank_promo_rate: 0.32


#### ACF

In [22]:
weekly = trends["aggregated_series"]
weekly_clean = weekly.dropna(subset=["log_total_liters_sold", "mean_log_price"])

acf_demand = eda.analyze_autocorrelation(
    weekly_clean,
    value_col="log_total_liters_sold",
    max_lags=52,
)

acf_price = eda.analyze_autocorrelation(
    weekly_clean,
    value_col="mean_log_price",
    max_lags=52,
)

lags = acf_demand["lags"]
acf_vals_d = acf_demand["acf_values"]
acf_vals_p = acf_price["acf_values"]

def acf_at(acf_vals, lag):
    return float(acf_vals[lag]) if lag < len(acf_vals) else np.nan

for lag in [1, 4, 13, 26, 52]:
    print(
        f"lag {lag}: demand ACF={acf_at(acf_vals_d, lag):.2f}, "
        f"price ACF={acf_at(acf_vals_p, lag):.2f}"
    )

lag 1: demand ACF=0.96, price ACF=0.98
lag 4: demand ACF=0.93, price ACF=0.94
lag 13: demand ACF=0.78, price ACF=0.84
lag 26: demand ACF=0.60, price ACF=0.68
lag 52: demand ACF=0.32, price ACF=0.33


#### Features temporales

In [23]:
df_feat = eda.build_temporal_features(
    df_imputed_prices,
    demand_col="log_liters_sold",
    promo_col="on_promo",
    season_periods=[52, 26, 13],
    lag_weeks=[1, 2, 4],
    rolling_windows=[4, 13], 
)
df_feat.head()

,category_code,upc_code,product_description,pack_size_text,units_per_case,product_item_code,store_code,week_id,units_sold,units_per_deal,...,miss_lag_2,lag_4_log_liters_sold,miss_lag_4,rolling_mean_4_log_liters_sold,rolling_median_4_log_liters_sold,miss_roll_4,rolling_mean_13_log_liters_sold,rolling_median_13_log_liters_sold,miss_roll_13,promo_intensity_store_week
0,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,166,2.0,1.0,...,1,0.000000,1,0.000000,0.000000,1,0.000000,0.000000,1,0.0
1,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,167,12.0,1.0,...,1,0.000000,1,2.835232,2.835232,0,2.835232,2.835232,0,0.75
2,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,168,13.0,1.0,...,0,0.000000,1,3.731112,3.731112,0,3.731112,3.731112,0,0.2
3,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,169,6.0,1.0,...,0,0.000000,1,4.056419,4.626992,0,4.056419,4.626992,0,0.0
4,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,170,4.0,1.0,...,0,2.835232,0,4.025776,4.280418,0,4.025776,4.280418,0,0.5


##### New features created

| Variable                                     | Type  | Description |
|----------------------------------------------|-------|-------------|
| `week_rank`                                  | int   | Sequential week index without gaps (1, 2, 3, …), sorted by `week_id`. Used as continuous time for trend and trigonometric features. |
| `sin_52`, `cos_52`                           | float | Sine and cosine components with period 52 weeks: capture **smooth annual seasonality** without discretizing time into dummies. |
| `sin_26`, `cos_26`                           | float | Seasonality with period 26 weeks (half-year) for semi-annual cycles. |
| `sin_13`, `cos_13`                           | float | Seasonality with period 13 weeks (quarterly), useful for quarter-level patterns. |
| `weeks_since_first_seen_upc`                 | int   | For each `upc_code`, number of weeks since its **first appearance** (0 on first week, 1 on the next, etc.). Models the **global product lifecycle**. |
| `weeks_since_first_seen_store_upc`           | int   | Same as above but per (`store_code`, `upc_code`) pair: product lifecycle **within that specific store**. |
| `lag_1_log_liters_sold`                      | float | `log_liters_sold` 1 week ago for the same (`store_code`, `upc_code`). Captures short-term demand inertia. |
| `lag_2_log_liters_sold`                      | float | Value 2 weeks ago; extends short/medium-term memory. |
| `lag_4_log_liters_sold`                      | float | Value 4 weeks ago; typically reflects monthly patterns. |
| `rolling_mean_4_log_liters_sold`             | float | 4-week rolling mean of `log_liters_sold` (historical window) per (`store_code`, `upc_code`). Smooths week-to-week noise. |
| `rolling_median_4_log_liters_sold`           | float | 4-week rolling median: more robust to outliers than the mean. |
| `rolling_mean_13_log_liters_sold` / `median` | float | 13-week (quarter) rolling mean/median; useful for underlying quarterly patterns. |
| `promo_intensity_store_week`                 | float | For each (`store_code`, `week_id`) pair, **share of UPCs on promotion** (0 to 1). Measures the store's overall promotional pressure that week. |

### Generic notation

- `lag_K_log_liters_sold`: log-demand \(K\) weeks ago for the same series (store × UPC).
- `rolling_mean_W_log_liters_sold`: mean over the last \(W\) weeks.
- `rolling_median_W_log_liters_sold`: median over the last \(W\) weeks.

#### Features Competitors

In [24]:
df_comp = eda.build_competitive_features(df_feat)
df_comp.head()

,category_code,upc_code,product_description,pack_size_text,units_per_case,product_item_code,store_code,week_id,units_sold,units_per_deal,...,neighbor_promo_share_sw_cat,n_same_brand_neighbors_sw_cat,same_brand_neighbor_promo_share_sw_cat,lag1_neighbor_mean_log_liters_sold,lag1_same_brand_neighbor_mean_log_liters_sold,roll4_neighbor_mean_log_liters_sold,store_category_upc_count_static,n_new_neighbors_13w,share_new_neighbors_13w,same_brand_upc_count_store_cat_static
0,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,166,2.0,1.0,...,0.0,2,0.0,NaN,NaN,NaN,5,2,1.0,3
1,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,167,12.0,1.0,...,1.0,2,1.0,3.587271,3.587271,3.587271,5,2,1.0,3
2,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,168,13.0,1.0,...,0.5,2,0.5,5.581763,5.581763,4.584517,5,2,1.0,3
3,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,169,6.0,1.0,...,0.0,2,0.0,5.125158,5.125158,4.764731,5,2,1.0,3
4,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,170,4.0,1.0,...,0.5,2,0.5,3.842684,3.842684,4.534219,5,2,1.0,3


##### New competitor features created

| Variable                                        | Type  | Description |
|-------------------------------------------------|-------|-------------|
| `n_neighbors_sw_cat`                            | int   | Number of neighbors in the same (`store_code`, `week_id`, `category_code`), excluding the focal UPC (`j != i`). |
| `neighbor_promo_share_sw_cat`                   | float | Share of neighbors on promotion in the same (`store_code`, `week_id`, `category_code`), excluding self. Uses only `on_promo`. |
| `n_same_brand_neighbors_sw_cat`                 | int   | Number of same-brand neighbors in the same store-week-category, excluding self (`brand_family_norm_j == brand_family_norm_i`). |
| `same_brand_neighbor_promo_share_sw_cat`        | float | Share of same-brand neighbors on promotion in the same store-week-category, excluding self. |
| `lag1_neighbor_mean_log_liters_sold`            | float | Mean `lag_1_log_liters_sold` across valid neighbors (`miss_lag_1 == 0`) in the same store-week-category, excluding self. |
| `roll4_neighbor_mean_log_liters_sold`           | float | Mean `rolling_mean_4_log_liters_sold` across valid neighbors (`miss_roll_4 == 0`) in the same store-week-category, excluding self. |
| `lag1_same_brand_neighbor_mean_log_liters_sold` | float | Mean lag-1 demand across valid same-brand neighbors (`miss_lag_1 == 0`), excluding self. |
| `store_category_upc_count_static`               | int   | Number of distinct UPCs in the same (`store_code`, `category_code`) modeled universe. |
| `same_brand_upc_count_store_cat_static`         | int   | Number of same-brand UPCs in the same (`store_code`, `category_code`), excluding the focal UPC. |
| `n_new_neighbors_13w`                           | int   | Number of neighbors considered “new” in-store, defined as `weeks_since_first_seen_store_upc <= 13`, excluding self. |
| `share_new_neighbors_13w`                       | float | Share of neighbors considered “new” in-store (`weeks_since_first_seen_store_upc <= 13`), excluding self. |

### Generic notation

- `sw_cat`: grouping by (`store_code`, `week_id`, `category_code`).
- `store_cat_static`: grouping by (`store_code`, `category_code`) over the modeled sample.
- `same_brand`: neighbors with `brand_family_norm_j == brand_family_norm_i`.
- `new_13w`: neighbors with `weeks_since_first_seen_store_upc <= 13`.
- All neighbor aggregates exclude self (`j != i`).
- For share features, if denominator is zero, the raw feature is undefined (`NaN`) before any optional imputation.

In [25]:
panel_stats = eda.analyze_panel_balance(df_comp)

print("Hypothetical full panel =", panel_stats["n_full_panel"])
print("Real density =", panel_stats["n_rows"], "/", panel_stats["n_full_panel"])
print("Real density (%) =", panel_stats["density_pct"])

Hypothetical full panel = 4390890
Real density = 532965 / 4390890
Real density (%) = 12.138


In [26]:
width_stats = eda.analyze_store_week_width(df_comp)

per_sw = width_stats["per_store_week_width"]
per_sw["n_upcs_store_week"].describe()

count    20718.000000
mean        25.724732
std         17.122039
min          1.000000
25%          9.000000
50%         26.000000
75%         38.000000
max         80.000000
Name: n_upcs_store_week, dtype: float64

### Correlation

In [27]:
sanity = eda.analyze_log_price_log_demand(
    df_comp,
    price_col="log_price_per_liter",
    demand_col="log_liters_sold",
    promo_col="on_promo",
)

print("Correlations log(price) – log(demand):")
print("  global:              ", round(sanity["corr_global"], 3))
print("  within (store×UPC):  ", round(sanity["corr_within_store_upc"], 3))
print("  within non‑promo:    ", round(sanity["corr_global_non_promo"], 3))
print("n_obs:", sanity["n_obs"], "| n_obs_non_promo:", sanity["n_obs_non_promo"])

Correlations log(price) – log(demand):
  global:               -0.505
  within (store×UPC):   -0.451
  within non‑promo:     -0.442
n_obs: 463722 | n_obs_non_promo: 339523


In [28]:
baseline = eda.analyze_baseline_elasticity_ols(
    df_comp,
    price_col="log_price_per_liter",
    demand_col="log_liters_sold",
    week_col="week_id",
)

print("OLS naive (no FE):     elasticity aprox.", round(baseline["ols_naive"]["elasticity"], 3))
print("OLS 2-way FE:           elasticity aprox.", round(baseline["ols_2way_fe"]["elasticity"], 3))
print("R² naive:", round(baseline["ols_naive"]["r_squared"], 4))
print("R² 2-way FE:", round(baseline["ols_2way_fe"]["r_squared"], 4))
print("n_obs naive:", baseline["ols_naive"]["n_obs"], "| n_obs 2-way FE:", baseline["ols_2way_fe"]["n_obs"])

OLS naive (no FE):     elasticity aprox. -1.984
OLS 2-way FE:           elasticity aprox. -2.97
R² naive: 0.2285
R² 2-way FE: 0.2352
n_obs naive: 463722 | n_obs 2-way FE: 463722


#### Final Filter

In [29]:
real = (
    (df_comp["is_imputed_calendar_row"] == 0)
    & (df_comp["is_global_gap_week"] == 0)
    & df_comp["log_liters_sold"].notna()
    & np.isfinite(df_comp["log_liters_sold"])
    & df_comp["log_price_per_liter"].notna()
    & np.isfinite(df_comp["log_price_per_liter"])
)
df_real = df_comp.loc[real].copy()

#### Missing competitors

In [30]:
competitive_cols = [
    "lag1_neighbor_mean_log_liters_sold",
    "roll4_neighbor_mean_log_liters_sold",
    "lag1_same_brand_neighbor_mean_log_liters_sold",
]
for col in competitive_cols:
    miss = df_real[col].isna()
    df_real[f"miss_{col}"] = miss.astype("int8")
    df_real[col] = df_real[col].fillna(0.0)
    
df_real.head()

,category_code,upc_code,product_description,pack_size_text,units_per_case,product_item_code,store_code,week_id,units_sold,units_per_deal,...,lag1_neighbor_mean_log_liters_sold,lag1_same_brand_neighbor_mean_log_liters_sold,roll4_neighbor_mean_log_liters_sold,store_category_upc_count_static,n_new_neighbors_13w,share_new_neighbors_13w,same_brand_upc_count_store_cat_static,miss_lag1_neighbor_mean_log_liters_sold,miss_roll4_neighbor_mean_log_liters_sold,miss_lag1_same_brand_neighbor_mean_log_liters_sold
0,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,166,2.0,1.0,...,0.000000,0.000000,0.000000,5,2,1.0,3,1,1,1
1,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,167,12.0,1.0,...,3.587271,3.587271,3.587271,5,2,1.0,3,0,0,0
2,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,168,13.0,1.0,...,5.581763,5.581763,4.584517,5,2,1.0,3,0,0,0
3,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,169,6.0,1.0,...,5.125158,5.125158,4.764731,5,2,1.0,3,0,0,0
4,27,3410015306,MILLER GENUINE DRFT,24/12O,1,9450340,5,170,4.0,1.0,...,3.842684,3.842684,4.534219,5,2,1.0,3,0,0,0


#### Nans?

In [31]:
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None
):
    display(eda.analyze_nans(df_real))

,n_nans,pct_nans
category_code,0,0.0
upc_code,0,0.0
units_per_case,0,0.0
product_item_code,0,0.0
store_code,0,0.0
week_id,0,0.0
units_sold,0,0.0
units_per_deal,0,0.0
total_price,0,0.0
gross_margin_pct,0,0.0


## Features selection for DL

In [32]:
columns_to_keep =  [
    "store_code",
    "upc_code",
    "brand_family_norm",
    "style_segment_norm", 
    "category_code",
    "week_id",

    "log_liters_sold",
    "log_price_per_liter",

    "on_promo",
    "week_rank",
    "sin_52", "cos_52",
    "sin_26", "cos_26",
    "sin_13", "cos_13",

    "weeks_since_first_seen_upc",
    "weeks_since_first_seen_store_upc",
    "liters_per_upc",

    "lag_1_log_liters_sold",
    "lag_2_log_liters_sold",
    "lag_4_log_liters_sold",
    "rolling_mean_4_log_liters_sold",
    "rolling_mean_13_log_liters_sold",

    "miss_lag_1",
    "miss_lag_2",
    "miss_lag_4",
    "miss_roll_4",
    "miss_roll_13",

    "promo_intensity_store_week",

    'n_neighbors_sw_cat',
    'neighbor_promo_share_sw_cat', 
    
    'n_same_brand_neighbors_sw_cat',
    'same_brand_neighbor_promo_share_sw_cat',

    'lag1_neighbor_mean_log_liters_sold',
    'lag1_same_brand_neighbor_mean_log_liters_sold',
    'roll4_neighbor_mean_log_liters_sold',

    'miss_lag1_neighbor_mean_log_liters_sold',
    'miss_roll4_neighbor_mean_log_liters_sold',
    'miss_lag1_same_brand_neighbor_mean_log_liters_sold',

    'store_category_upc_count_static',
    'same_brand_upc_count_store_cat_static',
    
    'n_new_neighbors_13w',
    'share_new_neighbors_13w', 
]
df_final = df_real[columns_to_keep].copy()
df_final.head()

,store_code,upc_code,brand_family_norm,style_segment_norm,category_code,week_id,log_liters_sold,log_price_per_liter,on_promo,week_rank,...,lag1_neighbor_mean_log_liters_sold,lag1_same_brand_neighbor_mean_log_liters_sold,roll4_neighbor_mean_log_liters_sold,miss_lag1_neighbor_mean_log_liters_sold,miss_roll4_neighbor_mean_log_liters_sold,miss_lag1_same_brand_neighbor_mean_log_liters_sold,store_category_upc_count_static,same_brand_upc_count_store_cat_static,n_new_neighbors_13w,share_new_neighbors_13w
0,5,3410015306,MILLER,DRAFT,27,166,2.835232,0.341957,0.0,76,...,0.000000,0.000000,0.000000,1,1,1,5,3,2,1.0
1,5,3410015306,MILLER,DRAFT,27,167,4.626992,0.254875,1.0,77,...,3.587271,3.587271,3.587271,0,0,0,5,3,2,1.0
2,5,3410015306,MILLER,DRAFT,27,168,4.707034,0.254875,0.0,78,...,5.581763,5.581763,4.584517,0,0,0,5,3,2,1.0
3,5,3410015306,MILLER,DRAFT,27,169,3.933844,0.341957,0.0,79,...,5.125158,5.125158,4.764731,0,0,0,5,3,2,1.0
4,5,3410015306,MILLER,DRAFT,27,170,3.528379,0.341957,1.0,80,...,3.842684,3.842684,4.534219,0,0,0,5,3,2,1.0


## Save

In [33]:
saver = DominickDataSaver()
saver.save("elasticity_dataset.csv", df_final)

## Colineallity 

In [33]:
# For the benchmark, we need to know if these variables are colinear.
cols_collinearity = [
    "on_promo",
    "week_rank",
    "sin_52", "cos_52",
    "sin_26", "cos_26",
    "sin_13", "cos_13",
    "weeks_since_first_seen_upc",
    "weeks_since_first_seen_store_upc",
    "liters_per_upc",
    "lag_1_log_liters_sold",
    "lag_2_log_liters_sold",
    "lag_4_log_liters_sold",
    "rolling_mean_4_log_liters_sold",
    "rolling_mean_13_log_liters_sold",
    "miss_lag_1",
    "miss_lag_2",
    "miss_lag_4",
    "miss_roll_4",
    "miss_roll_13",
    "promo_intensity_store_week",
    "n_neighbors_sw_cat",
    "neighbor_promo_share_sw_cat",
    "n_same_brand_neighbors_sw_cat",
    "same_brand_neighbor_promo_share_sw_cat",
    "lag1_neighbor_mean_log_liters_sold",
    "lag1_same_brand_neighbor_mean_log_liters_sold",
    "roll4_neighbor_mean_log_liters_sold",
    "miss_lag1_neighbor_mean_log_liters_sold",
    "miss_roll4_neighbor_mean_log_liters_sold",
    "miss_lag1_same_brand_neighbor_mean_log_liters_sold",
    "store_category_upc_count_static",
    "same_brand_upc_count_store_cat_static",
    "n_new_neighbors_13w",
    "share_new_neighbors_13w",
]

res = eda.analyze_collinearity(df_final, cols_collinearity)

display(pd.Series(res["summary"]))
display(res["high_corr_pairs_top"])
display(res["vif_table_top"])
display(res["focus_corr_matrix"])

n_rows_input                 463722
n_rows_used                  463722
n_features_input                 36
n_features_used_for_vif          36
corr_method                spearman
corr_threshold                  0.8
vif_threshold                   5.0
n_high_corr_pairs                11
n_high_vif_features              21
dtype: object

,var_a,var_b,corr,abs_corr
10,n_new_neighbors_13w,share_new_neighbors_13w,0.987589,0.987589
7,n_same_brand_neighbors_sw_cat,same_brand_upc_count_store_cat_static,0.976177,0.976177
2,weeks_since_first_seen_upc,weeks_since_first_seen_store_upc,0.970965,0.970965
6,n_neighbors_sw_cat,store_category_upc_count_static,0.965475,0.965475
5,rolling_mean_4_log_liters_sold,rolling_mean_13_log_liters_sold,0.940037,0.940037
0,week_rank,weeks_since_first_seen_upc,0.893714,0.893714
8,lag1_neighbor_mean_log_liters_sold,roll4_neighbor_mean_log_liters_sold,0.882072,0.882072
1,week_rank,weeks_since_first_seen_store_upc,0.857419,0.857419
3,lag_1_log_liters_sold,rolling_mean_4_log_liters_sold,0.834993,0.834993
9,lag1_same_brand_neighbor_mean_log_liters_sold,miss_lag1_same_brand_neighbor_mean_log_liters_...,-0.834546,0.834546


,variable,vif,flag_high_vif
14,rolling_mean_4_log_liters_sold,183.782996,True
28,roll4_neighbor_mean_log_liters_sold,136.355415,True
26,lag1_neighbor_mean_log_liters_sold,126.423282,True
15,rolling_mean_13_log_liters_sold,105.679707,True
8,weeks_since_first_seen_upc,85.244260,True
33,same_brand_upc_count_store_cat_static,70.966140,True
32,store_category_upc_count_static,68.990524,True
22,n_neighbors_sw_cat,68.990084,True
24,n_same_brand_neighbors_sw_cat,68.821448,True
9,weeks_since_first_seen_store_upc,62.327132,True


,lag1_neighbor_mean_log_liters_sold,lag1_same_brand_neighbor_mean_log_liters_sold,lag_1_log_liters_sold,lag_2_log_liters_sold,lag_4_log_liters_sold,liters_per_upc,miss_lag1_neighbor_mean_log_liters_sold,miss_lag1_same_brand_neighbor_mean_log_liters_sold,n_neighbors_sw_cat,n_new_neighbors_13w,...,promo_intensity_store_week,roll4_neighbor_mean_log_liters_sold,rolling_mean_13_log_liters_sold,rolling_mean_4_log_liters_sold,same_brand_upc_count_store_cat_static,share_new_neighbors_13w,store_category_upc_count_static,week_rank,weeks_since_first_seen_store_upc,weeks_since_first_seen_upc
lag1_neighbor_mean_log_liters_sold,1.000000,0.505624,0.307394,0.255093,0.213773,0.273247,-0.330496,-0.326332,0.410410,0.260532,...,0.037009,0.882072,0.279457,0.295364,0.257589,0.242885,0.434420,-0.147854,-0.155290,-0.161288
lag1_same_brand_neighbor_mean_log_liters_sold,0.505624,1.000000,0.330777,0.283119,0.247639,0.300142,-0.229452,-0.834546,0.444026,0.204457,...,0.018771,0.482262,0.331875,0.336543,0.732580,0.183169,0.446101,-0.103991,-0.085295,-0.095661
lag_1_log_liters_sold,0.307394,0.330777,1.000000,0.712421,0.586976,0.521340,-0.214923,-0.291079,0.184713,0.107652,...,0.015902,0.279182,0.794779,0.834993,0.280730,0.100701,0.191805,-0.063988,-0.004777,-0.017697
lag_2_log_liters_sold,0.255093,0.283119,0.712421,1.000000,0.594009,0.498981,-0.157698,-0.258345,0.179169,0.096944,...,-0.020697,0.265197,0.766947,0.814457,0.264640,0.089423,0.184441,-0.053805,0.007529,-0.005950
lag_4_log_liters_sold,0.213773,0.247639,0.586976,0.594009,1.000000,0.479287,-0.113630,-0.232591,0.173657,0.082190,...,0.033335,0.242246,0.735497,0.763022,0.252573,0.074552,0.176742,-0.050482,0.017196,0.002175
liters_per_upc,0.273247,0.300142,0.521340,0.498981,0.479287,1.000000,-0.059553,-0.324036,0.307692,0.123953,...,0.017550,0.296143,0.670145,0.632972,0.277684,0.111611,0.308273,-0.085719,-0.042570,-0.059361
miss_lag1_neighbor_mean_log_liters_sold,-0.330496,-0.229452,-0.214923,-0.157698,-0.113630,-0.059553,1.000000,0.274942,-0.156269,-0.058630,...,-0.005147,-0.235769,-0.103016,-0.145389,-0.103494,-0.053949,-0.134117,0.032686,0.016587,0.025838
miss_lag1_same_brand_neighbor_mean_log_liters_sold,-0.326332,-0.834546,-0.291079,-0.258345,-0.232591,-0.324036,0.274942,1.000000,-0.473470,-0.190081,...,-0.002190,-0.326917,-0.310059,-0.310146,-0.767987,-0.167866,-0.459897,0.102961,0.079217,0.091463
n_neighbors_sw_cat,0.410410,0.444026,0.184713,0.179169,0.173657,0.307692,-0.156269,-0.473470,1.000000,0.374661,...,-0.048332,0.449542,0.241166,0.227299,0.529923,0.302850,0.965475,-0.189531,-0.125962,-0.167699
n_new_neighbors_13w,0.260532,0.204457,0.107652,0.096944,0.082190,0.123953,-0.058630,-0.190081,0.374661,1.000000,...,-0.107326,0.285665,0.130015,0.130501,0.202671,0.987589,0.378290,-0.464855,-0.426149,-0.433518
